In [ ]:
import polars as pl
from pathlib import Path
import geopandas as gpd
import h3
from google.colab import drive
from shapely.geometry import Point
import ee
import numpy as np
from shapely import STRtree
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor


In [ ]:
drive.mount('/content/drive')

In [ ]:
base = Path('/content/drive/MyDrive')

In [ ]:
df = pl.read_csv(base / 'renewably_wind' / 'final_df_processed.csv')
df.describe()

In [ ]:
def explode_to_children(df: pl.DataFrame, target_res: int) -> pl.DataFrame:
    """Split each H3 cell into children at target_res."""
    rows = []
    for h3_index in df["h3_index"].to_list():
        for child in h3.cell_to_children(h3_index, res=target_res):
            lat, lon = h3.cell_to_latlon(child)
            rows.append({
                "h3_index": child,
                "parent_h3": h3_index,
                "lat": lat,
                "lon": lon,
            })
    return pl.DataFrame(rows)

df = explode_to_children(df, 9)
df.head()

## Wind turbine geojson

In [ ]:
turbine_gdf = gpd.read_file(base / 'renewably_wind' / 'wind_us_farms.geojson')

In [ ]:
turbine_gdf = turbine_gdf.to_crs(epsg=4326)
centroids = turbine_gdf.geometry.to_crs(3857).centroid.to_crs(4326)
turbine_gdf['longitude'] = centroids.x
turbine_gdf['latitude'] = centroids.y


In [ ]:
turbine_gdf = turbine_gdf[['fid', 'longitude', 'latitude']]
turbine_gdf.shape

In [ ]:
res = 9

turbine_cells = (
    pl.from_records(
        [
            (h3.latlon_to_cell(lat, lon, res),)
            for lat, lon in zip(
                turbine_gdf["latitude"],
                turbine_gdf["longitude"],
            )
        ],
        schema=["h3_index"],
        orient="row"
    )
    .unique()
)


In [ ]:
df = df.join(
    turbine_cells.with_columns(pl.lit(True).alias("has_turbine")),
    on="h3_index",
    how="left",
).with_columns(pl.col("has_turbine").fill_null(False))

In [ ]:
df.head()

In [ ]:
df['has_turbine'].value_counts()


In [ ]:
n = df['has_turbine'].value_counts().filter(~pl.col("has_turbine"))["count"].item() * .07

true_df = df.filter(pl.col("has_turbine"))
false_df = df.filter(~pl.col("has_turbine")).sample(n=n, seed=42)

df_balanced = pl.concat([true_df, false_df])

In [ ]:
df_balanced.head()
df_balanced['has_turbine'].value_counts()


In [ ]:
df_balanced.head()

### Download Data Sets

In [ ]:
ee.Authenticate()
ee.Initialize(project='renewably')

In [ ]:
cells = gpd.read_file(base / 'renewably_wind' / 'h3_cells.csv')

In [ ]:
cells.columns


In [ ]:
gdf = gpd.GeoDataFrame(
    cells, 
    geometry=gpd.points_from_xy(cells["lon"], cells["lat"]),
    crs="EPSG:4326"
)

gdf.to_csv(base / 'renewably_wind' / 'points.csv', index=False)

In [ ]:
points = gpd.read_file(base / 'renewably_wind' / 'points.csv')

In [ ]:
# Code to generate terrain features on earth engine

points = ee.FeatureCollection('projects/renewably/assets/points')

# Sources
nlcd = ee.ImageCollection('USGS/NLCD_RELEASES/2021_REL/NLCD').first()
nlcd_landcover = nlcd.select(['landcover'])
nlcd_impervious = nlcd.select(['impervious'])
elevation = ee.Image('USGS/SRTMGL1_003').select(['elevation'])
slope_img = ee.Terrain.slope(elevation).rename('slope')

# ERA5 wind speed
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') \
    .filterDate('2020-01-01', '2023-12-31') \
    .select(['u_component_of_wind_10m', 'v_component_of_wind_10m'])

def wind_speed(img):
    return img.expression(
        'sqrt(u**2 + v**2)',
        {'u': img.select('u_component_of_wind_10m'),
         'v': img.select('v_component_of_wind_10m')}
    ).rename('wind_speed')

mean_wind = era5.map(wind_speed).mean()

# Soil taxonomy
soil = ee.Image('OpenLandMap/SOL/SOL_GRTGROUP_USDA-SOILTAX_C/v01') \
    .select(['grtgroup']).rename('soil_type')

# --- 30m stack (native resolutions, no reproject) ---
stack_30m = (nlcd_landcover
    .addBands(nlcd_impervious)
    .addBands(elevation)
    .addBands(slope_img)
    .addBands(soil)
    .addBands(mean_wind)
)

results_30m = stack_30m.reduceRegions(
    collection=points,
    reducer=ee.Reducer.first(),
    scale=30
)

task1 = ee.batch.Export.table.toAsset(
    collection=results_30m,
    description='terrain_30m_v2',
    assetId='projects/renewably/assets/terrain_30m_v2'
)
task1.start()

# --- Coarse stack (rasterized vectors + GHSL only) ---
crs = nlcd_landcover.projection()

padus = ee.FeatureCollection('USGS/GAP/PAD-US/v20/designation')
protected = padus.reduceToImage(
    properties=[],
    reducer=ee.Reducer.countEvery()
).gt(0).rename('protected_area').unmask(0) \
 .reproject(crs=crs, scale=250)

wdpa = ee.FeatureCollection('WCMC/WDPA/current/polygons')
in_wdpa = wdpa.reduceToImage(
    properties=[],
    reducer=ee.Reducer.countEvery()
).gt(0).rename('in_wdpa').unmask(0) \
 .reproject(crs=crs, scale=250)

ghsl = ee.Image('JRC/GHSL/P2023A/GHS_POP/2020') \
    .select('population_count') \
    .rename('pop_density') \
    .updateMask(ee.Image('JRC/GHSL/P2023A/GHS_POP/2020')
        .select('population_count').gte(0)) \
    .reproject(crs=crs, scale=250)

stack_coarse = (protected
    .addBands(in_wdpa)
    .addBands(ghsl)
)

results_coarse = stack_coarse.reduceRegions(
    collection=points,
    reducer=ee.Reducer.first(),
    scale=250
)

task2 = ee.batch.Export.table.toAsset(
    collection=results_coarse,
    description='terrain_coarse_v2',
    assetId='projects/renewably/assets/terrain_coarse_v2'
)
task2.start()

In [ ]:
terrain_30m = gpd.read_file(base / 'renewably_wind' / 'terrain_30m2.csv')
terrain_coarse = gpd.read_file(base / 'renewably_wind' / 'terrain_coarse2.csv')


In [189]:
df = pl.from_pandas(terrain_30m.merge(terrain_coarse[['h3_index', 'protected_area', 'in_wdpa', 'pop_density']],
                  on='h3_index', how='left'))

In [190]:
df.head()

system:index,elevation,h3_index,has_turbine,impervious,landcover,lon_lat,slope,soil_type,wind_speed,.geo,protected_area,in_wdpa,pop_density
str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""0000000000000000fc4a""","""451""","""89277280b17ffff""","""false""","""""","""""","""{""type"":""Point"",""coordinates"":…","""0.0""","""""","""0.9011865979726744""","""{""type"":""Point"",""coordinates"":…","""1""","""1""","""0.0"""
"""0000000000000002d88f""","""320""","""89270ec43d7ffff""","""false""","""""","""""","""{""type"":""Point"",""coordinates"":…","""0.0""","""""","""1.1457143436516062""","""{""type"":""Point"",""coordinates"":…","""0""","""0""","""0.0"""
"""0000000000000001e32b""","""1735""","""8912d99206fffff""","""false""","""""","""""","""{""type"":""Point"",""coordinates"":…","""21.41647572859848""","""16""","""0.7400736873409682""","""{""type"":""Point"",""coordinates"":…","""0""","""0""","""0.0"""
"""00000000000000019f72""","""1496""","""8912caa8ccbffff""","""false""","""""","""""","""{""type"":""Point"",""coordinates"":…","""7.708171728720353""","""16""","""0.9522970071545882""","""{""type"":""Point"",""coordinates"":…","""0""","""1""","""0.0"""
"""0000000000000001220d""","""470""","""89277289d87ffff""","""false""","""""","""""","""{""type"":""Point"",""coordinates"":…","""5.716593910214294""","""216""","""0.8879737990087687""","""{""type"":""Point"",""coordinates"":…","""1""","""1""","""0.0"""


In [191]:
df = df.rename({
    'landcover': 'land_type',
    'elevation': 'elevation_m',
    'slope': 'slope_deg'
})

In [192]:
df = df.drop(["system:index"])

In [193]:
import json

# Parse the 'lon_lat' JSON column and extract longitude and latitude using Polars expressions
df = df.with_columns([
    pl.col("lon_lat").map_elements(
        lambda x: json.loads(x)["coordinates"][0] if x is not None else None,
        return_dtype=pl.Float64,
    ).alias("lon"),
    pl.col("lon_lat").map_elements(
        lambda x: json.loads(x)["coordinates"][1] if x is not None else None,
        return_dtype=pl.Float64,
    ).alias("lat"),
])

# Drop 'lon_lat' and '.geo' columns if present
df = df.drop(["lon_lat", ".geo"])

In [194]:
df.head()

elevation_m,h3_index,has_turbine,impervious,land_type,slope_deg,soil_type,wind_speed,protected_area,in_wdpa,pop_density,lon,lat
str,str,str,str,str,str,str,str,str,str,str,f64,f64
"""451""","""89277280b17ffff""","""false""","""""","""""","""0.0""","""""","""0.9011865979726744""","""1""","""1""","""0.0""",-90.118762,48.105045
"""320""","""89270ec43d7ffff""","""false""","""""","""""","""0.0""","""""","""1.1457143436516062""","""0""","""0""","""0.0""",-94.817723,49.317283
"""1735""","""8912d99206fffff""","""false""","""""","""""","""21.41647572859848""","""16""","""0.7400736873409682""","""0""","""0""","""0.0""",-115.902538,49.000972
"""1496""","""8912caa8ccbffff""","""false""","""""","""""","""7.708171728720353""","""16""","""0.9522970071545882""","""0""","""1""","""0.0""",-114.337861,49.001231
"""470""","""89277289d87ffff""","""false""","""""","""""","""5.716593910214294""","""216""","""0.8879737990087687""","""1""","""1""","""0.0""",-90.396694,48.102098


In [195]:
df = df.filter(~(pl.col("wind_speed") == ""))
df = df.filter(~(pl.col("pop_density") == ""))
df = df.filter(~(pl.col("protected_area") == ""))
df = df.filter(~(pl.col("in_wdpa") == ""))
df = df.filter(~(pl.col("land_type") == ""))
df = df.filter(~(pl.col("impervious") == ""))
df = df.filter(~(pl.col("soil_type") == ""))



In [196]:
# Fix nulls
df = df.with_columns([
    pl.col("elevation_m").cast(pl.Float64).fill_null(0),
    pl.col("slope_deg").cast(pl.Float64).fill_null(0), 
    pl.col("soil_type").cast(pl.Int32).fill_null(0),
    pl.col("wind_speed").cast(pl.Float64).fill_null(0),
    pl.col("protected_area").cast(pl.Float64).fill_null(0),
    pl.col("in_wdpa").cast(pl.Float64).fill_null(0),
    pl.col("pop_density").cast(pl.Float64).fill_null(0),
])

In [197]:
# Convert "has_turbine" column to boolean
# (in polars, strings "true"/"false" must first be mapped to boolean values before casting)
df = df.with_columns(
    pl.when(pl.col("has_turbine") == "true")
      .then(True)
      .when(pl.col("has_turbine") == "false")
      .then(False)
      .otherwise(None)
      .alias("has_turbine")
)
df.describe()

statistic,elevation_m,h3_index,has_turbine,impervious,land_type,slope_deg,soil_type,wind_speed,protected_area,in_wdpa,pop_density,lon,lat
str,f64,str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",261926.0,"""261926""",261926.0,"""261926""","""261926""",261926.0,261926.0,261926.0,261926.0,261926.0,261926.0,261926.0,261926.0
"""null_count""",0.0,"""0""",0.0,"""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",764.844078,null,0.252144,null,null,5.414795,213.776177,1.265802,0.157651,0.067928,0.30931,-99.293127,39.171275
"""std""",676.43925,null,null,null,null,6.402979,120.920819,0.558707,0.364415,0.251622,3.069545,12.575262,5.298934
"""min""",-83.0,"""8912c904327ffff""",0.0,"""0""","""11""",0.0,1.0,0.231191,0.0,0.0,0.0,-124.669817,25.161432
"""25%""",259.0,null,null,null,null,1.677003,93.0,0.848511,0.0,0.0,0.0,-108.344798,35.002539
"""50%""",512.0,null,null,null,null,3.061186,272.0,1.153727,0.0,0.0,0.0,-99.141298,39.53373
"""75%""",1167.0,null,null,null,null,5.965766,290.0,1.544418,0.0,0.0,0.0,-90.701964,43.400141
"""max""",4192.0,"""8948f6ccd73ffff""",1.0,"""99""","""95""",76.176107,434.0,6.097871,1.0,1.0,272.28923,-67.013957,49.323998


In [198]:
# Encodings 

# NLCD Land Cover — group into meaningful categories for turbine siting
landtype_encoding = {
    "0":  0,   # No data / unclassified
    "11": 1,   # Open water
    "12": 1,   # Perennial ice/snow
    "21": 2,   # Developed, open space
    "22": 3,   # Developed, low intensity
    "23": 4,   # Developed, medium intensity
    "24": 5,   # Developed, high intensity
    "31": 6,   # Barren land
    "41": 7,   # Deciduous forest
    "42": 8,   # Evergreen forest
    "43": 9,   # Mixed forest
    "52": 10,  # Shrub/scrub
    "71": 11,  # Grassland/herbaceous
    "81": 12,  # Pasture/hay
    "82": 13,  # Cultivated crops
    "90": 14,  # Woody wetlands
    "95": 15,  # Emergent herbaceous wetlands
}

# Impervious  encoding
impervious_encoding = {
    "0":  0,   # Unclassified
    "20": 1,   # Primary road
    "21": 2,   # Secondary road
    "22": 3,   # Tertiary road
    "23": 4,   # Thinned road
    "24": 5,   # Non-road non-energy impervious
    "25": 6,   # Microsoft buildings
    "26": 7,   # LCMAP impervious
    "27": 0,   # Wind turbines → mask to unclassified (LEAKAGE)
    "28": 8,   # Well pads
    "29": 9,   # Other energy production
}

df = df.with_columns(
    pl.col("impervious")
        .fill_null("0")
        .cast(pl.UInt8)
        .replace_strict(impervious_encoding, default=0),
    pl.col("land_type")
        .fill_null("0")
        .cast(pl.UInt8)
        .replace_strict(landtype_encoding, default=0)
)


In [203]:
# Build code-to-order mapping from the class table
soil_code_to_order = {
    0: 0,  # NODATA
    # Alfisols (-alfs)
    1: 1, 2: 1, 4: 1, 6: 1, 7: 1, 9: 1, 10: 1, 11: 1, 12: 1, 13: 1,
    14: 1, 15: 1, 16: 1, 17: 1, 18: 1, 19: 1, 25: 1, 26: 1, 27: 1,
    28: 1, 29: 1, 30: 1, 31: 1, 32: 1, 38: 1, 39: 1, 41: 1, 42: 1,
    43: 1, 44: 1, 45: 1, 46: 1,
    # Andisols (-ands)
    50: 2, 58: 2, 59: 2, 61: 2, 63: 2, 64: 2, 74: 2, 75: 2, 76: 2,
    77: 2, 80: 2,
    # Aridisols (-ids)
    82: 3, 83: 3, 85: 3, 86: 3, 87: 3, 89: 3, 90: 3, 92: 3, 93: 3,
    94: 3, 96: 3, 97: 3, 98: 3, 99: 3, 100: 3, 101: 3, 102: 3, 103: 3,
    104: 3, 105: 3, 107: 3, 110: 3, 111: 3, 112: 3, 113: 3, 114: 3,
    115: 3, 116: 3,
    # Entisols (-ents)
    118: 4, 119: 4, 120: 4, 121: 4, 122: 4, 123: 4, 124: 4, 126: 4,
    131: 4, 133: 4, 134: 4, 135: 4, 136: 4, 137: 4, 138: 4, 139: 4,
    140: 4, 141: 4, 142: 4, 143: 4, 144: 4, 145: 4, 146: 4, 147: 4,
    148: 4, 149: 4, 153: 4, 154: 4, 155: 4,
    # Histosols (-ists)
    179: 5, 180: 5, 181: 5, 182: 5, 183: 5, 184: 5, 185: 5, 186: 5,
    189: 5, 190: 5, 191: 5, 196: 5, 201: 5, 202: 5, 203: 5,
    # Inceptisols (-epts)
    206: 6, 207: 6, 208: 6, 209: 6, 210: 6, 212: 6, 213: 6, 215: 6,
    216: 6, 217: 6, 218: 6, 219: 6, 220: 6, 221: 6, 222: 6, 225: 6,
    226: 6, 228: 6, 229: 6, 230: 6, 231: 6, 233: 6, 234: 6, 235: 6,
    245: 6, 246: 6, 247: 6, 248: 6, 249: 6, 250: 6, 251: 6, 252: 6,
    253: 6, 254: 6, 255: 6, 256: 6,
    # Mollisols (-olls)
    268: 7, 269: 7, 270: 7, 271: 7, 272: 7, 273: 7, 274: 7, 275: 7,
    276: 7, 277: 7, 278: 7, 279: 7, 280: 7, 283: 7, 284: 7, 285: 7,
    286: 7, 287: 7, 289: 7, 290: 7, 291: 7, 292: 7, 294: 7, 296: 7,
    297: 7, 298: 7, 299: 7, 300: 7, 301: 7, 302: 7, 303: 7, 306: 7,
    307: 7, 308: 7, 309: 7, 310: 7, 311: 7, 312: 7,
    # Spodosols (-ods)
    342: 8, 343: 8, 345: 8, 348: 8, 349: 8, 350: 8, 351: 8, 353: 8,
    354: 8, 356: 8, 357: 8, 358: 8, 367: 8, 368: 8,
    # Ultisols (-ults)
    370: 9, 371: 9, 372: 9, 373: 9, 374: 9, 375: 9, 376: 9, 377: 9,
    378: 9, 381: 9, 385: 9, 387: 9, 388: 9, 389: 9, 390: 9, 391: 9,
    396: 9, 399: 9, 401: 9,
    # Vertisols (-erts)
    403: 10, 405: 10, 406: 10, 409: 10, 410: 10, 412: 10, 413: 10,
    414: 10, 415: 10, 417: 10, 418: 10, 419: 10, 420: 10, 422: 10,
    424: 10, 429: 10, 430: 10, 431: 10, 432: 10, 433: 10,
}

df = df.with_columns(
    pl.col("soil_type")
      .cast(pl.Int32)
      .replace_strict(soil_code_to_order, default=0)
      .cast(pl.UInt8)
)

In [216]:
df.describe()

statistic,elevation_m,h3_index,has_turbine,impervious,land_type,slope_deg,soil_type,protected_area,in_wdpa,pop_density,lon,lat,road_dist_km,transmission_line_dist_km,wind_speed,airport_dist_km
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",261926.0,"""261926""",261926.0,261926.0,261926.0,261926.0,261926.0,261926.0,261926.0,261926.0,261926.0,261926.0,261926.0,261926.0,261926.0,261926.0
"""null_count""",0.0,"""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",764.844078,null,0.252144,0.025507,10.365966,5.414795,0.881577,0.157651,0.067928,0.30931,-99.293127,39.171275,45.536132,7.492378,3.24101,15.091411
"""std""",676.43925,null,null,0.392585,2.770778,6.402979,0.323109,0.364415,0.251622,3.069545,12.575262,5.298934,44.944309,10.225064,1.006711,10.345433
"""min""",-83.0,"""8912c904327ffff""",0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-124.669817,25.161432,0.00035,0.00001,1.143677,0.013582
"""25%""",259.0,null,null,0.0,9.0,1.677003,1.0,0.0,0.0,0.0,-108.344798,35.002539,12.957472,1.393535,2.347674,7.707735
"""50%""",512.0,null,null,0.0,11.0,3.061186,1.0,0.0,0.0,0.0,-99.141298,39.53373,30.228337,3.737915,3.213805,12.811828
"""75%""",1167.0,null,null,0.0,13.0,5.965766,1.0,0.0,0.0,0.0,-90.701964,43.400141,63.190743,9.018832,4.196416,19.797206
"""max""",4192.0,"""8948f6ccd73ffff""",1.0,9.0,15.0,76.176107,1.0,1.0,1.0,272.28923,-67.013957,49.323998,302.182166,109.433714,6.620546,87.16485


In [205]:
df.shape

(261926, 13)

In [213]:
def proximity_to_transmission_line(df: pl.DataFrame) -> pl.DataFrame:
    """Calculate the distance to the nearest transmission line for each point."""
    transmission_lines = gpd.read_file(base / 'renewably_wind' / 'us_transmission_lines.geojson')
    transmission_lines = transmission_lines.to_crs(epsg=5070)

    points = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(df['lon'].to_list(), df['lat'].to_list()),
        crs="EPSG:4326"
    ).to_crs(epsg=5070)

    tree = STRtree(transmission_lines.geometry.values)
    point_geoms = points.geometry.values

    indices, distances_m = tree.query_nearest(point_geoms, return_distance=True)
    input_idx = indices[0]

    min_distances = np.full(len(points), np.inf)
    np.minimum.at(min_distances, input_idx, distances_m)
    distances_km = min_distances / 1000.0

    return df.with_columns(
        pl.Series("transmission_line_dist_km", distances_km)
    )



def proximity_to_road(df: pl.DataFrame) -> pl.DataFrame:
    """Calculate distance to nearest road for each point"""
    road_lines = gpd.read_file(base / 'renewably_wind' / 'tl_2023_us_primaryroads.shp').set_crs(epsg=4269)
    road_lines = road_lines.to_crs(epsg=5070)

    points = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(df['lon'].to_list(), df['lat'].to_list()),
        crs="EPSG:4326"
    ).to_crs(epsg=5070)

    tree = STRtree(road_lines.geometry.values)
    point_geoms = points.geometry.values

    indices, distances_m = tree.query_nearest(point_geoms, return_distance=True)
    input_idx = indices[0]

    min_distances = np.full(len(points), np.inf)
    np.minimum.at(min_distances, input_idx, distances_m)
    distances_km = min_distances / 1000.0

    return df.with_columns(
        pl.Series("road_dist_km", distances_km)
    )


def proximity_to_airport(df: pl.DataFrame) -> pl.DataFrame:
    airports = gpd.read_file(base / 'renewably_wind' / 'airports.geojson')
    airports = airports.to_crs(epsg=5070)

    points = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(df['lon'].to_list(), df['lat'].to_list()),
        crs="EPSG:4326"
    ).to_crs(epsg=5070)

    tree = STRtree(airports.geometry.values)
    point_geoms = points.geometry.values

    indices, distances_m = tree.query_nearest(point_geoms, return_distance=True)
    input_idx = indices[0]

    min_distances = np.full(len(points), np.inf)
    np.minimum.at(min_distances, input_idx, distances_m)
    distances_km = min_distances / 1000.0

    return df.with_columns(
        pl.Series("airport_dist_km", distances_km)
    )

def wind_speed(df: pl.DataFrame) -> pl.DataFrame:
    wind_speeds = pd.read_csv(base / 'renewably_wind' / 'us_wind_speed_dataset_2022.csv')

    knn = KNeighborsRegressor(n_neighbors=5, weights="distance")
    knn.fit(wind_speeds[["lat", "lon"]], wind_speeds["annual_mean_wind_speed"])

    pdf = df.to_pandas()
    for col in ["wind_speed", "annual_mean_wind_speed"]:
        if col in pdf.columns:
            pdf = pdf.drop(columns=[col])

    pdf["wind_speed"] = knn.predict(pdf[["lat", "lon"]].rename(columns={"lon": "lon"}))

    return pl.from_pandas(pdf)
    

In [209]:
df = proximity_to_road(df)

In [210]:
df = proximity_to_transmission_line(df)

In [211]:
df = wind_speed(df)

In [214]:
df = proximity_to_airport(df)

## Data clean up

In [222]:
df.describe()

statistic,elevation_m,h3_index,has_turbine,impervious,land_type,slope_deg,soil_type,protected_area,in_wdpa,pop_density,lon,lat,road_dist_km,transmission_line_dist_km,wind_speed,airport_dist_km
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",261872.0,"""261872""",261872.0,261872.0,261872.0,261872.0,261872.0,261872.0,261872.0,261872.0,261872.0,261872.0,261872.0,261872.0,261872.0,261872.0
"""null_count""",0.0,"""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",764.921916,null,0.25199,0.025512,10.367072,5.415336,0.881568,0.15768,0.067942,0.308859,-99.294817,39.171076,45.541193,7.493427,3.240915,15.092747
"""std""",676.469669,null,null,0.392625,2.769992,6.403366,0.323119,0.364441,0.251646,3.065845,12.574861,5.299112,44.945693,10.225791,1.006722,10.345605
"""min""",-83.0,"""8912c904327ffff""",0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-124.669817,25.161432,0.00035,0.00001,1.143677,0.013582
"""25%""",259.0,null,null,0.0,9.0,1.67734,1.0,0.0,0.0,0.0,-108.346136,35.00249,12.963625,1.393885,2.34756,7.709913
"""50%""",512.0,null,null,0.0,11.0,3.061343,1.0,0.0,0.0,0.0,-99.142266,39.532776,30.2305,3.738675,3.213563,12.813624
"""75%""",1167.0,null,null,0.0,13.0,5.966194,1.0,0.0,0.0,0.0,-90.704305,43.400212,63.196094,9.020369,4.196321,19.79867
"""max""",4192.0,"""8948f6ccd73ffff""",1.0,9.0,15.0,76.176107,1.0,1.0,1.0,272.28923,-67.013957,49.323998,302.182166,109.433714,6.620546,87.16485


In [220]:
# make sure data is correct, land types
df = df.filter(~(pl.col("land_type").is_in([5]) & pl.col("has_turbine")))

In [221]:
df = df.with_columns(pl.col("pop_density").clip(lower_bound=0))

In [224]:
df.write_csv(base / 'renewably_wind' / 'final_df_processed_with_proximity.csv')